In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

PROCESSED = '/content/drive/MyDrive/MSc_Project/data/processed/'
OUTPUTS   = '/content/drive/MyDrive/MSc_Project/outputs/'
POWERBI   = '/content/drive/MyDrive/MSc_Project/powerbi/'

import os
os.makedirs(POWERBI, exist_ok=True)

# FILE 1: MAIN DATASET - actual vs predicted + gold overlay

# Load feature-engineered dataset (has all indicators)
df = pd.read_csv(PROCESSED + 'usd_lkr_gold_features.csv',
                 index_col='Date', parse_dates=True)

# Load predictions
preds_df = pd.read_csv(OUTPUTS + '06_all_predictions.csv',
                       index_col=0, parse_dates=True)

# Load ensemble predictions (best 3 models)
preds_df['ensemble'] = (preds_df['arima'] + preds_df['arimax'] + preds_df['xgboost']) / 3

# Load GARCH confidence intervals
garch_vol = preds_df['garch_vol']
preds_df['upper_ci'] = preds_df['ensemble'] + 1.96 * garch_vol
preds_df['lower_ci'] = preds_df['ensemble'] - 1.96 * garch_vol

# Build main export - test period only (2022–2026)
main_export = pd.DataFrame({
    'Date':              preds_df.index,
    'actual_returns':    preds_df['actual'],
    'arima_pred':        preds_df['arima'],
    'arimax_pred':       preds_df['arimax'],
    'xgboost_pred':      preds_df['xgboost'],
    'lstm_pred':         preds_df['lstm'],
    'gru_pred':          preds_df['gru'],
    'ensemble_pred':     preds_df['ensemble'],
    'upper_ci':          preds_df['upper_ci'],
    'lower_ci':          preds_df['lower_ci'],
    'garch_vol':         preds_df['garch_vol'],
})

main_export.to_csv(POWERBI + '01_predictions.csv', index=False)
print(f"File 1 saved: 01_predictions.csv | {len(main_export)} rows")

# FILE 2: FULL HISTORICAL DATA - LKR price + gold price (1975–2026)

historical = df[['lkr_close', 'lkr_open', 'lkr_high', 'lkr_low',
                  'gold_close', 'gold_open']].copy()
historical = historical.reset_index()
historical.columns = ['Date', 'lkr_close', 'lkr_open', 'lkr_high',
                       'lkr_low', 'gold_close', 'gold_open']

historical.to_csv(POWERBI + '02_historical_prices.csv', index=False)
print(f"File 2 saved: 02_historical_prices.csv | {len(historical)} rows")

# FILE 3: TECHNICAL INDICATORS (test period)

test_start = '2022-01-01'
indicators = df[df.index >= test_start][[
    'lkr_close',
    'lkr_rsi_14',
    'lkr_macd',
    'lkr_macd_signal',
    'lkr_macd_diff',
    'lkr_bb_high',
    'lkr_bb_low',
    'lkr_bb_width',
    'lkr_sma_30',
    'lkr_ema_10',
    'gold_close',
    'gold_rsi_14',
    'gold_bb_width',
]].copy().reset_index()

indicators.to_csv(POWERBI + '03_technical_indicators.csv', index=False)
print(f"File 3 saved: 03_technical_indicators.csv | {len(indicators)} rows")

# FILE 4: MODEL COMPARISON TABLE

model_results = pd.DataFrame([
    {'Model': 'ARIMA',                        'Type': 'Individual', 'RMSE': 0.007943, 'MAE': 0.002872},
    {'Model': 'GARCH(1,1)',                   'Type': 'Individual', 'RMSE': 0.007943, 'MAE': 0.002872},
    {'Model': 'XGBoost',                      'Type': 'Individual', 'RMSE': 0.007960, 'MAE': 0.002923},
    {'Model': 'ARIMAX',                       'Type': 'Individual', 'RMSE': 0.008019, 'MAE': 0.003293},
    {'Model': 'Best 3 Ensemble',              'Type': 'Ensemble',   'RMSE': 0.008056, 'MAE': 0.002978},
    {'Model': 'Statistical + XGBoost',        'Type': 'Ensemble',   'RMSE': 0.008056, 'MAE': 0.002978},
    {'Model': 'Statistical only',             'Type': 'Ensemble',   'RMSE': 0.008065, 'MAE': 0.003067},
    {'Model': 'Hybrid Ensemble (weighted)',   'Type': 'Ensemble',   'RMSE': 0.008395, 'MAE': 0.003742},
    {'Model': 'Equal weight ensemble',        'Type': 'Ensemble',   'RMSE': 0.010627, 'MAE': 0.007134},
    {'Model': 'LSTM',                         'Type': 'Individual', 'RMSE': 0.014709, 'MAE': 0.008308},
    {'Model': 'Deep learning only',           'Type': 'Ensemble',   'RMSE': 0.018331, 'MAE': 0.015037},
    {'Model': 'GRU',                          'Type': 'Individual', 'RMSE': 0.038416, 'MAE': 0.032279},
])

model_results.to_csv(POWERBI + '04_model_results.csv', index=False)
print(f"File 4 saved: 04_model_results.csv | {len(model_results)} rows")

# FILE 5: ABLATION STUDY

ablation = pd.DataFrame([
    {'Configuration': 'With gold features',          'RMSE': 0.008060, 'MAE': 0.003040},
    {'Configuration': 'Without gold (ARIMA only)',   'RMSE': 0.008050, 'MAE': 0.002910},
])

ablation.to_csv(POWERBI + '05_ablation_study.csv', index=False)
print(f"File 5 saved: 05_ablation_study.csv | {len(ablation)} rows")


print("\nAll Power BI data files saved to:")
print("  MSc_Project/powerbi/")
print("\nFiles ready:")
print("  01_predictions.csv         - actual vs predicted + confidence intervals")
print("  02_historical_prices.csv   - full 1975-2026 LKR and gold price history")
print("  03_technical_indicators.csv - RSI, MACD, Bollinger Bands (test period)")
print("  04_model_results.csv       - model comparison table")
print("  05_ablation_study.csv      - ablation study results")

Mounted at /content/drive
File 1 saved: 01_predictions.csv | 1076 rows
File 2 saved: 02_historical_prices.csv | 12646 rows
File 3 saved: 03_technical_indicators.csv | 1106 rows
File 4 saved: 04_model_results.csv | 12 rows
File 5 saved: 05_ablation_study.csv | 2 rows

All Power BI data files saved to:
  MSc_Project/powerbi/

Files ready:
  01_predictions.csv         - actual vs predicted + confidence intervals
  02_historical_prices.csv   - full 1975-2026 LKR and gold price history
  03_technical_indicators.csv - RSI, MACD, Bollinger Bands (test period)
  04_model_results.csv       - model comparison table
  05_ablation_study.csv      - ablation study results
